# Embedding Methods for RAG

A survey of the embedding techniques used to turn text into vectors, from 1990s bag-of-words counting up to modern transformer and API-based embeddings. Each section has:

1. **What it is / how it works**
2. **`pip install`** for the library it needs
3. **Runnable code** against a shared sample corpus

At the end, all methods are compared side by side on the same two similarity checks, so the difference between "counts overlapping words" and "understands meaning" is visible in actual numbers, not just theory.

| # | Method | Type | Library |
|---|--------|------|---------|
| 1 | Bag of Words | sparse, count-based | `scikit-learn` |
| 2 | N-gram Bag of Words | sparse, count-based | `scikit-learn` |
| 3 | TF-IDF | sparse, weighted count | `scikit-learn` |
| 4 | Hashing Vectorizer | sparse, count-based (fixed size) | `scikit-learn` |
| 5 | Word2Vec | dense, static word vectors | `gensim` |
| 6 | GloVe | dense, static word vectors (pretrained) | `gensim` |
| 7 | FastText | dense, static word vectors (subword-aware) | `gensim` |
| 8 | Doc2Vec | dense, static document vectors | `gensim` |
| 9 | BERT (raw) | dense, contextual word vectors | `transformers` |
| 10 | Sentence-Transformers (SBERT) | dense, contextual sentence vectors | `sentence-transformers` |
| 11 | Instruction embeddings (BGE/E5-style) | dense, contextual, instruction-tuned | `sentence-transformers` |
| 12 | OpenAI Embeddings API | dense, hosted | `openai` |
| 13 | Vertex AI Embeddings API | dense, hosted | `google-cloud-aiplatform` |
| 14 | Cohere Embed API | dense, hosted | `cohere` |


In [ ]:
%pip install -q scikit-learn numpy pandas
%pip install -q gensim nltk
%pip install -q transformers torch sentence-transformers
# optional — only needed for the hosted-API sections (§12-14)
%pip install -q openai cohere google-cloud-aiplatform


## Sample Corpus

The same 12 sentences run through every method below, so results are directly comparable. Sentence 1 and sentence 2 are semantically related (share almost no exact words); sentence 12 is unrelated to everything else — a good stress test for "does this method understand meaning, or just count matching words."


In [ ]:
corpus = [
    "Retrieval-augmented generation combines a retriever with a language model.",   # 0
    "The system fetches relevant passages from a knowledge store before answering.",  # 1 - related to 0, few shared words
    "A vector database stores embeddings for fast similarity search.",              # 2
    "Word2Vec learns dense word vectors from a large text corpus.",                 # 3
    "TF-IDF weights words by how unique they are to a document.",                   # 4
    "Bag of words ignores word order and grammar entirely.",                        # 5
    "Sentence transformers produce dense embeddings for whole sentences.",          # 6
    "BERT uses a transformer encoder to build contextual word representations.",    # 7
    "FastText represents words as bags of character n-grams.",                      # 8
    "Doc2Vec extends word2vec to learn embeddings for entire documents.",           # 9
    "Cosine similarity measures the angle between two vectors.",                    # 10
    "The cat sat on the warm windowsill in the afternoon sun.",                     # 11 - unrelated
]

RELATED_PAIR = (0, 1)      # semantically related, lexically different
UNRELATED_PAIR = (0, 11)   # unrelated

corpus_vectors: dict[str, "object"] = {}  # method name -> (n_docs, dim) array, filled in as we go

for i, s in enumerate(corpus):
    print(f"[{i}] {s}")


## 1. Bag of Words (BoW)

**What it is:** count how many times each vocabulary word appears in a document — a sparse vector of raw word counts, with no notion of order, grammar, or meaning. `CountVectorizer` builds the vocabulary from the corpus and produces one row per document.

**When to use it:** simple keyword-matching search, spam/topic classification baselines, or as the sparse half of a hybrid retrieval system (paired with dense embeddings — see the previous notebook's BM25 section, which is BoW's smarter cousin).

```
pip install scikit-learn
```


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(corpus)

print(f"vocabulary size: {len(bow_vectorizer.vocabulary_)}")
print(f"matrix shape: {bow_matrix.shape}  (documents x vocabulary)")
print(f"sentence 0 as counts: {bow_matrix[0].toarray().sum()} total word occurrences, "
      f"{ (bow_matrix[0].toarray() > 0).sum() } distinct words")

corpus_vectors["bow"] = bow_matrix.toarray()


## 2. N-gram Bag of Words

**What it is:** the same counting idea, but over contiguous word sequences (bigrams, trigrams) instead of single words. This captures some local word order ("not good" vs "good") that unigram BoW throws away entirely, at the cost of a much larger, sparser vocabulary.

**When to use it:** when short local phrases carry meaning single words don't (negation, compound terms like "New York"), and you're staying in sparse/keyword territory rather than moving to embeddings.

```
pip install scikit-learn
```


In [ ]:
ngram_vectorizer = CountVectorizer(ngram_range=(1, 2))  # unigrams + bigrams
ngram_matrix = ngram_vectorizer.fit_transform(corpus)

print(f"unigram-only vocabulary: {len(bow_vectorizer.vocabulary_)}")
print(f"unigram+bigram vocabulary: {len(ngram_vectorizer.vocabulary_)}")
print("example bigrams:", [t for t in ngram_vectorizer.vocabulary_ if " " in t][:8])

corpus_vectors["bow_ngram"] = ngram_matrix.toarray()


## 3. TF-IDF

**What it is:** Term Frequency-Inverse Document Frequency reweights raw counts: words that appear often *within* a document but *rarely across* the corpus get a high score, while words that appear everywhere (the, is, a — or in a technical corpus, words like "embedding" if every document mentions it) get pushed toward zero. This is what makes TF-IDF far more useful than raw BoW for ranking/search.

**When to use it:** classic keyword search (it's literally half of BM25's formula), document similarity baselines, or feature extraction for simple ML classifiers — still a strong, cheap baseline to compare fancier methods against.

```
pip install scikit-learn
```


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

# show the highest-weighted terms for sentence 4 (about TF-IDF itself)
feature_names = tfidf_vectorizer.get_feature_names_out()
row = tfidf_matrix[4].toarray()[0]
top_terms = sorted(zip(feature_names, row), key=lambda x: x[1], reverse=True)[:5]
print("top TF-IDF terms for sentence 4:", top_terms)

corpus_vectors["tfidf"] = tfidf_matrix.toarray()


## 4. Hashing Vectorizer

**What it is:** instead of building an explicit vocabulary dictionary (which has to be stored and grows without bound on streaming/huge corpora), hash each token directly into one of a fixed number of buckets. Same counting idea as BoW, but memory-bounded and able to handle unseen vocabulary at inference time without retraining — at the cost of hash collisions (two different words landing in the same bucket) and losing the ability to map a feature back to its original word.

**When to use it:** very large or streaming corpora where fitting/storing a vocabulary is impractical (e.g. online learning pipelines).

```
pip install scikit-learn
```


In [ ]:
from sklearn.feature_extraction.text import HashingVectorizer

hashing_vectorizer = HashingVectorizer(n_features=2**10, alternate_sign=False)
hashing_matrix = hashing_vectorizer.fit_transform(corpus)

print(f"matrix shape: {hashing_matrix.shape}  (fixed width, no vocabulary stored, no .fit needed)")
corpus_vectors["hashing"] = hashing_matrix.toarray()


## 5. Word2Vec

**What it is:** a shallow neural network learns a dense vector per *word* such that words appearing in similar contexts end up close together — trained either by predicting a word from its surrounding context (CBOW) or predicting the context from a word (Skip-gram). This was the first embedding method to capture real semantic/analogical structure (`king - man + woman ≈ queen`).

Word2Vec produces one vector per *word*, not per sentence — to embed a sentence, average (or otherwise pool) its word vectors. This is crude (it discards word order and grammar) but works surprisingly well as a baseline.

**When to use it:** rarely chosen fresh today for RAG — sentence-transformers (below) dominate — but still relevant for lightweight/offline use cases, or as the conceptual base that later methods build on.

```
pip install gensim nltk
```


In [ ]:
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec

nltk.download("punkt_tab", quiet=True)
tokenized_corpus = [word_tokenize(s.lower()) for s in corpus]

# A real Word2Vec model needs millions of words to learn good vectors — this tiny corpus is
# only enough to prove the mechanics work, not to produce meaningful vectors. For real use,
# load a pretrained model instead: gensim.downloader.load("word2vec-google-news-300")
w2v_model = Word2Vec(sentences=tokenized_corpus, vector_size=100, window=5, min_count=1, workers=1, seed=42)

def average_word_vectors(tokens: list[str], model, dim: int) -> np.ndarray:
    vectors = [model.wv[t] for t in tokens if t in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(dim)

w2v_matrix = np.array([average_word_vectors(toks, w2v_model, 100) for toks in tokenized_corpus])
print(f"word2vec matrix shape: {w2v_matrix.shape}")
corpus_vectors["word2vec_tiny"] = w2v_matrix


## 6. GloVe (Pretrained)

**What it is:** unlike Word2Vec's local context windows, GloVe (Global Vectors) factorizes a global word-word co-occurrence matrix built over the *entire* corpus — a different training objective that tends to land in a similar place in practice. GloVe is almost always used as a **pretrained** download (trained on Wikipedia/Common Crawl by Stanford) rather than trained from scratch, since it needs a huge corpus to be any good.

**When to use it:** same territory as Word2Vec — a lightweight, dependency-free (no GPU, no API) way to get real semantic word vectors when full sentence-transformer models are too heavy for the deployment target.

```
pip install gensim
```

⚠️ The download below pulls a real pretrained model (~66MB for the 50-dim Wikipedia vectors) — skip this cell if you're offline or don't want the download.


In [ ]:
import gensim.downloader as gensim_api

glove_model = gensim_api.load("glove-wiki-gigaword-50")  # 400k vocab, 50-dim, pretrained on Wikipedia + Gigaword
glove_dim = glove_model.vector_size

glove_matrix = np.array([average_word_vectors(toks, glove_model, glove_dim) for toks in tokenized_corpus])
print(f"glove matrix shape: {glove_matrix.shape}")

# classic sanity check: GloVe's most famous property — vector arithmetic captures analogies
print("king - man + woman ≈", glove_model.most_similar(positive=["king", "woman"], negative=["man"], topn=3))
corpus_vectors["glove"] = glove_matrix


## 7. FastText

**What it is:** extends Word2Vec by representing each word as a bag of character n-grams (e.g. `"where"` → `<wh, whe, her, ere, re>`, plus the whole word) and summing their vectors. This means FastText can produce a reasonable vector for a word it never saw during training, as long as it shares subword pieces with known words — Word2Vec and GloVe simply fail (or fall back to a zero/UNK vector) on out-of-vocabulary words.

**When to use it:** morphologically rich languages, noisy text with typos/rare word forms, or any domain (medical, legal, code identifiers) where you expect to hit words the training corpus never saw.

```
pip install gensim
```


In [ ]:
from gensim.models import FastText

fasttext_model = FastText(sentences=tokenized_corpus, vector_size=100, window=5, min_count=1, workers=1, seed=42)

oov_word = "retrievability"  # not in the tiny training corpus at all
print(f"'{oov_word}' in vocabulary? {oov_word in fasttext_model.wv.key_to_index}")
print(f"FastText still produces a vector via subwords: shape={fasttext_model.wv[oov_word].shape}")

fasttext_matrix = np.array([average_word_vectors(toks, fasttext_model, 100) for toks in tokenized_corpus])
corpus_vectors["fasttext_tiny"] = fasttext_matrix


## 8. Doc2Vec

**What it is:** extends Word2Vec's training objective with an extra "paragraph vector" that's trained alongside word vectors to be predictive of the words in its document — so you get one dense vector *per document* directly, instead of hand-averaging word vectors after the fact. `infer_vector` lets you embed brand-new text after training without retraining the whole model.

**When to use it:** when you specifically want trainable, corpus-adapted document embeddings without a transformer, e.g. clustering/deduplicating a large domain-specific document set where a generic pretrained sentence embedding might not capture domain-specific similarity well.

```
pip install gensim
```


In [ ]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

tagged_corpus = [TaggedDocument(words=toks, tags=[str(i)]) for i, toks in enumerate(tokenized_corpus)]
doc2vec_model = Doc2Vec(tagged_corpus, vector_size=100, window=5, min_count=1, workers=1, epochs=100, seed=42)

doc2vec_matrix = np.array([doc2vec_model.dv[str(i)] for i in range(len(corpus))])
print(f"doc2vec matrix shape: {doc2vec_matrix.shape}")

new_text = "Retrieving the right passages matters more than the model size."
inferred = doc2vec_model.infer_vector(word_tokenize(new_text.lower()))
print(f"inferred vector for unseen text: shape={inferred.shape}")

corpus_vectors["doc2vec_tiny"] = doc2vec_matrix


## 9. BERT (Raw Contextual Embeddings)

**What it is:** Word2Vec/GloVe give a word exactly one vector regardless of context — "bank" gets the same vector in "river bank" and "bank account." BERT (and transformers generally) run the whole sentence through self-attention layers, so each token's vector is *contextual* — it changes depending on the surrounding words. Sentence-transformers (next section) are BERT-family models specifically fine-tuned so *pooled* sentence vectors are directly comparable with cosine similarity; raw BERT wasn't trained for that, so its pooled output is a weaker similarity signal — shown here for the concept, not as something to use as-is in production.

**When to use it:** rarely used raw for embeddings in production (use sentence-transformers instead) — but understanding it is the basis for understanding *why* sentence-transformers exist and what they fix.

```
pip install transformers torch
```


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModel.from_pretrained("bert-base-uncased")
bert_model.eval()

def mean_pool_bert(text: str) -> np.ndarray:
    tokens = bert_tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        output = bert_model(**tokens)
    last_hidden = output.last_hidden_state[0]          # (seq_len, hidden_dim)
    return last_hidden.mean(dim=0).numpy()               # mean pooling over tokens

# Show contextuality: "bank" gets a different vector depending on meaning — Word2Vec cannot do this.
river_vec = mean_pool_bert("I sat by the river bank and watched the water.")
money_vec = mean_pool_bert("I deposited the check at the bank.")
from sklearn.metrics.pairwise import cosine_similarity as _cos
print(f"cosine(river-bank sentence, money-bank sentence) = {_cos([river_vec], [money_vec])[0][0]:.3f}  (same word, different sense)")

bert_matrix = np.array([mean_pool_bert(s) for s in corpus])
corpus_vectors["bert_raw"] = bert_matrix


## 10. Sentence-Transformers (SBERT)

**What it is:** BERT-family models fine-tuned (via a siamese/triplet network setup, on sentence-pair datasets like NLI and semantic-similarity benchmarks) specifically so that a single pooled vector per sentence is directly comparable with cosine similarity — the training objective raw BERT never had. This is the default choice for embeddings in most local/open-source RAG pipelines today, used throughout the previous two notebooks in this folder.

**When to use it:** the default for RAG unless you have a specific reason to use a hosted API (§12-14) — runs locally, no per-call cost, good quality/speed tradeoff at the `all-MiniLM-L6-v2` size.

```
pip install sentence-transformers
```


In [ ]:
from sentence_transformers import SentenceTransformer

sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
sbert_matrix = sbert_model.encode(corpus, normalize_embeddings=True)

print(f"sentence-transformers matrix shape: {sbert_matrix.shape}")
corpus_vectors["sbert"] = sbert_matrix


## 11. Instruction-Tuned Embeddings (BGE / E5-style)

**What it is:** the current generation of top open embedding models (BAAI's BGE, Microsoft's E5, etc.) add one more trick on top of the SBERT recipe: they're trained to expect a short **instruction prefix** that tells the model whether the text is a query or a passage being indexed (e.g. `"query: ..."` vs `"passage: ..."`), which measurably improves retrieval quality because it lets the model use different internal representations for the two roles — similar in spirit to Vertex AI's `RETRIEVAL_QUERY` vs `RETRIEVAL_DOCUMENT` task types from the previous notebook, but baked into the input text instead of an API parameter.

**When to use it:** when you want open/local embeddings but with quality closer to hosted APIs — these models are consistently at or near the top of the [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard) among open models.

```
pip install sentence-transformers
```


In [ ]:
bge_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

# BGE convention: prefix queries, leave passages/documents unprefixed
bge_query_prefix = "query: "
prefixed_corpus = [bge_query_prefix + s for s in corpus]  # treating the whole corpus as queries here just to compare vectors

bge_matrix = bge_model.encode(prefixed_corpus, normalize_embeddings=True)
print(f"BGE matrix shape: {bge_matrix.shape}")
corpus_vectors["bge_instruction"] = bge_matrix


## 12. OpenAI Embeddings API

**What it is:** a hosted embedding endpoint (`text-embedding-3-small` / `-large`) — no local model to run or GPU to manage, strong general-purpose quality, billed per token. The tradeoff versus local sentence-transformers/BGE models is a network call per request and an ongoing per-token cost instead of one-time compute.

```
pip install openai
```
Requires `OPENAI_API_KEY` in the environment — this cell is skipped automatically if it's not set.


In [ ]:
import os

if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI

    openai_client = OpenAI()
    response = openai_client.embeddings.create(model="text-embedding-3-small", input=corpus)
    openai_matrix = np.array([item.embedding for item in response.data])
    print(f"OpenAI matrix shape: {openai_matrix.shape}")
    corpus_vectors["openai"] = openai_matrix
else:
    print("OPENAI_API_KEY not set — skipping. Set it in the environment to run this section.")


## 13. Vertex AI Embeddings API

**What it is:** Google Cloud's hosted embedding model (`text-embedding-005`) — used throughout `vertex_rag_pipeline.ipynb` in this folder, shown here just for direct comparison against the other methods on the same corpus. See that notebook for the full production retrieval pipeline built on top of it.

```
pip install google-cloud-aiplatform
```
Requires a `PROJECT_ID` with the Vertex AI API enabled and `gcloud auth application-default login` run beforehand — this cell is skipped automatically if `PROJECT_ID` isn't set below.


In [ ]:
PROJECT_ID = ""  # TODO: set to your GCP project id to run this section
LOCATION = "us-central1"

if PROJECT_ID:
    import vertexai
    from vertexai.language_models import TextEmbeddingModel, TextEmbeddingInput

    vertexai.init(project=PROJECT_ID, location=LOCATION)
    vertex_model = TextEmbeddingModel.from_pretrained("text-embedding-005")
    inputs = [TextEmbeddingInput(s, "RETRIEVAL_DOCUMENT") for s in corpus]
    vertex_matrix = np.array([e.values for e in vertex_model.get_embeddings(inputs)])
    print(f"Vertex AI matrix shape: {vertex_matrix.shape}")
    corpus_vectors["vertex_ai"] = vertex_matrix
else:
    print("PROJECT_ID not set — skipping. See vertex_rag_pipeline.ipynb for full setup instructions.")


## 14. Cohere Embed API

**What it is:** another major hosted embedding provider (`embed-english-v3.0`), notable for an explicit `input_type` parameter (`search_document` vs `search_query` vs `classification` vs `clustering`) — the same query/document asymmetry idea as Vertex AI's task types and BGE's instruction prefixes, here exposed as a first-class API parameter with more granularity.

```
pip install cohere
```
Requires `COHERE_API_KEY` in the environment — this cell is skipped automatically if it's not set.


In [ ]:
if os.environ.get("COHERE_API_KEY"):
    import cohere

    cohere_client = cohere.Client(os.environ["COHERE_API_KEY"])
    response = cohere_client.embed(texts=corpus, model="embed-english-v3.0", input_type="search_document")
    cohere_matrix = np.array(response.embeddings)
    print(f"Cohere matrix shape: {cohere_matrix.shape}")
    corpus_vectors["cohere"] = cohere_matrix
else:
    print("COHERE_API_KEY not set — skipping. Set it in the environment to run this section.")


## Comparison

For every method that actually ran above, compute cosine similarity for:

- **Related pair** (sentences 0 & 1 — same topic, almost no shared vocabulary)
- **Unrelated pair** (sentences 0 & 11 — different topics entirely)

A method that "understands meaning" should score the related pair clearly higher than the unrelated one. A purely lexical method (BoW/TF-IDF) has no mechanism to do that unless the sentences happen to share words — which is exactly the gap dense embeddings close.


In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

rows = []
for name, matrix in corpus_vectors.items():
    related_sim = cosine_similarity([matrix[RELATED_PAIR[0]]], [matrix[RELATED_PAIR[1]]])[0][0]
    unrelated_sim = cosine_similarity([matrix[UNRELATED_PAIR[0]]], [matrix[UNRELATED_PAIR[1]]])[0][0]
    rows.append({
        "method": name,
        "dim": matrix.shape[1],
        "related_sim (0,1)": round(float(related_sim), 3),
        "unrelated_sim (0,11)": round(float(unrelated_sim), 3),
        "gap (related - unrelated)": round(float(related_sim - unrelated_sim), 3),
    })

comparison_df = pd.DataFrame(rows).sort_values("gap (related - unrelated)", ascending=False).reset_index(drop=True)
comparison_df


## Summary

| Method | Sparse/Dense | Contextual | Needs training | Typical RAG role |
|---|---|---|---|---|
| Bag of Words | sparse | no | fit on corpus | keyword baseline, not used alone in modern RAG |
| N-gram BoW | sparse | no (local order only) | fit on corpus | keyword baseline with light phrase awareness |
| TF-IDF | sparse | no | fit on corpus | sparse leg of hybrid search (paired with BM25/dense) |
| Hashing Vectorizer | sparse | no | none (stateless) | streaming/huge-corpus keyword search |
| Word2Vec | dense (word-level) | no | pretrained or trained | legacy baseline, lightweight offline use |
| GloVe | dense (word-level) | no | pretrained | legacy baseline, lightweight offline use |
| FastText | dense (word-level) | no | pretrained or trained | domain text with lots of OOV/rare words |
| Doc2Vec | dense (doc-level) | no | trained on corpus | domain-adapted clustering/dedup without a transformer |
| BERT (raw) | dense (contextual) | yes | pretrained | conceptual stepping stone to SBERT, not used as-is |
| Sentence-Transformers | dense (contextual) | yes | pretrained | **default local embedding choice for RAG** |
| BGE / E5 (instruction) | dense (contextual) | yes | pretrained | best open-model quality, when instruction prefixes are handled correctly |
| OpenAI / Vertex AI / Cohere | dense (contextual) | yes | pretrained (hosted) | production RAG at scale, no local GPU management |

**Rule of thumb:** for RAG retrieval quality, dense contextual embeddings (SBERT-family or a hosted API) are the right default — sparse methods (BoW/TF-IDF) mainly earn their place back as the *keyword* half of a hybrid dense+sparse retrieval setup (see `vertex_rag_pipeline.ipynb`'s BM25 section), not as the primary embedding. Static word embeddings (Word2Vec/GloVe/FastText) are largely legacy for RAG specifically, but still show up in lightweight NLP pipelines where a full transformer is too heavy.
